## New Experimentation

In [ ]:
pip install ultralytics

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize
import wandb
import timm
from ultralytics import YOLO
import time
import random
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [ ]:
def get_dataset(dataset_path):
    """Load dataset from folder structure. Designed to be lightweight for Colab.
    Expects folder layout: dataset_path/<class_name>/*.jpg
    """
    if dataset_path is None:
        dataset_path = os.environ.get('DATASET_PATH', './data')

    image_paths = []
    labels = []
    if not os.path.isdir(dataset_path):
        raise RuntimeError(f"Dataset path not found: {dataset_path}")

    class_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    for class_name in class_names:
        class_dir = os.path.join(dataset_path, class_name)
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(img_path)
                labels.append(class_to_idx[class_name])

    # Transform used in the project
    transform = torch.nn.Sequential()  # placeholder so the Dataset code in main still works
    from torchvision import transforms
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    class ImageDataset(torch.utils.data.Dataset):
        def __init__(self, image_paths, labels, transform=None):
            self.image_paths = image_paths
            self.labels = labels
            self.transform = transform

        def __len__(self):
            return len(self.image_paths)

        def __getitem__(self, idx):
            from PIL import Image
            img_path = self.image_paths[idx]
            image = Image.open(img_path).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                image = self.transform(image)
            return image, label

    dataset = ImageDataset(np.array(image_paths), np.array(labels), transform)
    print(f"Found {len(dataset)} images across {len(class_names)} classes: {class_names}")
    return dataset, class_names

In [ ]:
class YOLOv8Classifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=True, device='cpu'):
        super(YOLOv8Classifier, self).__init__()

        if pretrained:
            yolo_model = YOLO('yolo26n.pt')
        else:
            yolo_model = YOLO('yolo26n.yaml')

        self.model = yolo_model.model.model
        self.backbone_neck_layers = nn.ModuleList(list(self.model.children())[:-1])

        # Identify skip-connection layers
        self.save_indices = []
        for i, layer in enumerate(self.model.children()):
            if hasattr(layer, 'f') and layer.f != -1:
                if isinstance(layer.f, int):
                    self.save_indices.append(layer.f)
                elif isinstance(layer.f, (list, tuple)):
                    self.save_indices.extend(layer.f)

        # Infer output channels - use the specified device
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224).to(device)  # Changed to 224
            # Temporarily move backbone to device for inference
            self.backbone_neck_layers.to(device)
            features = self._extract_features(dummy_input)
            if isinstance(features, (list, tuple)):
                features = features[-1]
            self.feature_channels = features.shape[1]
            # Move back to CPU (will be moved to final device later)
            self.backbone_neck_layers.to('cpu')

        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(self.feature_channels, 64),  # Added hidden layer with 64 units
            nn.ReLU(),  # Activation for the hidden layer
            nn.Linear(64, num_classes)
        )

        # Freeze backbone initially
        for param in self.backbone_neck_layers.parameters():
            param.requires_grad = False

    def _extract_features(self, x):
        outputs = []
        for i, layer in enumerate(self.backbone_neck_layers):
            if hasattr(layer, 'f'):
                if layer.f != -1:
                    if isinstance(layer.f, int):
                        x = layer(outputs[layer.f]) if layer.f != -1 else layer(x)
                    else:
                        feature_list = [outputs[j] if j >= 0 else x for j in layer.f]
                        x = layer(feature_list)
                        if isinstance(x, (list, tuple)):
                            x = x[-1]
                else:
                    x = layer(x)
            else:
                x = layer(x)
            outputs.append(x)
        return x

    def forward(self, x):
        features = self._extract_features(x)
        if isinstance(features, (list, tuple)):
            features = features[-1]
        output = self.classifier(features)
        return output

    def unfreeze_backbone(self):
        for param in self.backbone_neck_layers.parameters():
            param.requires_grad = True


class YOLOv8Backbone(nn.Module):
    """Extract features from YOLOv8 Classifier - Aligned with training script"""
    def __init__(self, model):
        super().__init__()
        self.backbone_neck_layers = model.backbone_neck_layers
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flat = nn.Flatten()

        # Determine output dimension dynamically by running a dummy forward pass
        try:
            device_ = next(model.parameters()).device
            img_size = 224  # Changed to 224
            dummy = torch.zeros(1, 3, img_size, img_size, device=device_)

            self.eval()
            with torch.no_grad():
                feats = self._extract_features(dummy)
                pooled = self.pool(feats)
                flattened = self.flat(pooled)
                self.out_dim = flattened.shape[1]
        except Exception as e:
            # Fallback
            self.out_dim = model.feature_channels

    def _extract_features(self, x):
        outputs = []
        for i, layer in enumerate(self.backbone_neck_layers):
            if hasattr(layer, 'f'):
                if layer.f != -1:
                    if isinstance(layer.f, int):
                        x = layer(outputs[layer.f]) if layer.f != -1 else layer(x)
                    else:
                        feature_list = [outputs[j] if j >= 0 else x for j in layer.f]
                        x = layer(feature_list)
                        if isinstance(x, (list, tuple)):
                            x = x[-1]
                else:
                    x = layer(x)
            else:
                x = layer(x)
            outputs.append(x)
        return x

    def forward(self, x):
        features = self._extract_features(x)
        if isinstance(features, (list, tuple)):
            features = features[-1]
        pooled = self.pool(features)
        out = self.flat(pooled)
        return out


In [ ]:


def build_vit_tiny(num_classes, num_blocks_to_keep=8):
    vit = timm.create_model('vit_tiny_patch16_224', pretrained=False, num_classes=0)
    embed_dim = 192
    if num_blocks_to_keep < len(vit.blocks):
        vit.blocks = vit.blocks[:num_blocks_to_keep]

    class ViTWithHead(nn.Module):
        def __init__(self, vit_backbone, embed_dim, num_classes, dropout=0.1):
            super().__init__()
            self.vit = vit_backbone
            self.head = nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, num_classes)
            )

        def forward(self, x):
            x = self.vit.forward_features(x)
            cls_token = x[:, 0]
            logits = self.head(cls_token)
            return logits

    return ViTWithHead(vit, embed_dim, num_classes)


class ViTTinyBackbone(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vit = model.vit
        self.out_dim = 192

    def forward(self, x):
        features = self.vit.forward_features(x)
        cls_token = features[:, 0]
        return cls_token

In [ ]:
class YOLOViTFusion(nn.Module):
    def __init__(self, yolo_backbone, vit_backbone, num_classes, dropout=0.3):
        super().__init__()
        self.yolo = yolo_backbone
        self.vit = vit_backbone
        self.yolo_norm = nn.LayerNorm(self.yolo.out_dim)  # 256 for YOLO
        self.vit_norm = nn.LayerNorm(self.vit.out_dim)    # 192 for ViT
        fused_dim = self.yolo.out_dim + self.vit.out_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes)
        )
        self._init_classifier()

    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x, eval_mode=True):
        if eval_mode:
            with torch.no_grad():
                f_yolo = self.yolo(x)
                f_vit = self.vit(x)
        else:
            f_yolo = self.yolo(x)
            f_vit = self.vit(x)
        # Apply LayerNorm before concatenation
        f_yolo_norm = self.yolo_norm(f_yolo)
        f_vit_norm = self.vit_norm(f_vit)
        fused = torch.cat([f_yolo_norm, f_vit_norm], dim=1)
        out = self.classifier(fused)
        return out

In [ ]:
# Utility Functions
def calculate_additional_metrics(y_true, y_pred, y_prob, num_classes):
    """Calculate precision, recall, f1, and roc_auc"""
    metrics = {}

    # Macro averaged metrics
    metrics['precision'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
    metrics['f1'] = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # ROC AUC
    try:
        if num_classes == 2:
            metrics['roc_auc'] = roc_auc_score(y_true, y_prob[:, 1])
        else:
            y_true_bin = label_binarize(y_true, classes=range(num_classes))
            metrics['roc_auc'] = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
    except:
        metrics['roc_auc'] = 0.0

    return metrics


def plot_cm(cm, class_names, title):
    """Plot and save confusion matrix"""
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()
    return img_path


class EarlyStopping:
    def __init__(self, patience=7, min_delta=0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_val_acc = 0

    def __call__(self, val_acc):
        score = val_acc
        if self.best_score is None:
            self.best_score = score
            self.best_val_acc = val_acc
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_val_acc = val_acc
            self.counter = 0

In [ ]:
def load_pretrained_backbones(yolo_weights, vit_weights):
    num_classes = 3
    print("Building YOLOv8 Classifier...")
    yolo_model = YOLOv8Classifier(num_classes=num_classes, pretrained=True)

    if yolo_weights is not None and os.path.exists(yolo_weights):
        print(f"Loading YOLO weights from {yolo_weights} ...")
        yolo_checkpoint = torch.load(yolo_weights, map_location=device)
        if isinstance(yolo_checkpoint, dict):
            if 'model_state_dict' in yolo_checkpoint:
                state_dict = yolo_checkpoint['model_state_dict']
            elif 'state_dict' in yolo_checkpoint:
                state_dict = yolo_checkpoint['state_dict']
            else:
                state_dict = yolo_checkpoint
        else:
            state_dict = yolo_checkpoint

        new_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith('model.'):
                # Remap 'model.X' to 'detector_layers.X' as in the training script
                new_key = key.replace('model.', 'detector_layers.', 1)
                new_state_dict[new_key] = value
            else:
                # Include all other keys (including original classifier keys if any) for consistency
                # The YOLOv8Classifier now uses the actual Ultralytics Classify block
                new_state_dict[key] = value

        yolo_model.load_state_dict(new_state_dict, strict=False)
        print("✓ Loaded YOLO weights (remapped keys)")
    else:
        print("No YOLO weights provided or path not found; using pretrained ultralytics backbone state")

    print("Building ViT-Tiny...")
    vit_model = build_vit_tiny(num_classes, num_blocks_to_keep=8)
    if vit_weights is not None and os.path.exists(vit_weights):
        print(f"Loading ViT weights from {vit_weights} ...")
        vit_checkpoint = torch.load(vit_weights, map_location=device, weights_only=False)
        if isinstance(vit_checkpoint, dict):
            if 'model_state_dict' in vit_checkpoint:
                vit_model.load_state_dict(vit_checkpoint['model_state_dict'])
            elif 'state_dict' in vit_checkpoint:
                vit_model.load_state_dict(vit_checkpoint['state_dict'])
            else:
                vit_model.load_state_dict(vit_checkpoint)
        else:
            vit_model.load_state_dict(vit_checkpoint)
    else:
        print("No ViT weights provided or path not found; using random-initialized ViT (training recommended)")

    yolo_model.to(device).eval()
    vit_model.to(device).eval()

    yolo_backbone = YOLOv8Backbone(yolo_model)
    vit_backbone = ViTTinyBackbone(vit_model)

    for p in yolo_backbone.parameters():
        p.requires_grad = False
    for p in vit_backbone.parameters():
        p.requires_grad = False

    print("✓ Backbones ready")
    print(f"YOLO feature dim: {yolo_backbone.out_dim}")
    print(f"ViT feature dim: {vit_backbone.out_dim}")

    return yolo_backbone, vit_backbone

In [ ]:
"""
Clean Parameter Summary for YOLO26–ViT Fusion
Shows ONLY:
- YOLO Backbone
- ViT Backbone
- Fusion Classifier
"""

import torch
import torch.nn as nn

# =====================================================
# Helper Functions
# =====================================================

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



yolo_classifier = YOLOv8Classifier(
    num_classes=3,
    pretrained=True,
    device=device
).to(device)

yolo_backbone = YOLOv8Backbone(yolo_classifier)

# --- ViT Backbone ---
vit_classifier = build_vit_tiny(
    num_classes=3,
    num_blocks_to_keep=8
).to(device)

vit_backbone = ViTTinyBackbone(vit_classifier)


fusion_model = YOLOViTFusion(
    yolo_backbone,
    vit_backbone,
    num_classes=3,
    dropout=0.3
).to(device)


for p in fusion_model.yolo.parameters():
    p.requires_grad = False

for p in fusion_model.vit.parameters():
    p.requires_grad = False


yolo_params        = count_parameters(fusion_model.yolo)
vit_params         = count_parameters(fusion_model.vit)
classifier_params  = count_parameters(fusion_model.classifier)

total_params       = count_parameters(fusion_model)
trainable_params   = count_trainable_parameters(fusion_model)
frozen_params      = total_params - trainable_params



print("="*60)
print("YOLO26–ViT FUSION MODEL PARAMETER SUMMARY")
print("="*60)

print(f"YOLO Backbone        : {yolo_params:>12,} params   (FROZEN)")
print(f"ViT Backbone         : {vit_params:>12,} params   (FROZEN)")
print(f"Fusion Classifier    : {classifier_params:>12,} params   (TRAINABLE)")

print("-"*60)
print(f"Total Fusion Model   : {total_params:>12,} params")
print(f"Trainable Parameters : {trainable_params:>12,}")
print(f"Frozen Parameters    : {frozen_params:>12,}")
print("="*60)


YOLO26–ViT FUSION MODEL PARAMETER SUMMARY
YOLO Backbone        :    2,262,624 params   (FROZEN)
ViT Backbone         :    3,744,960 params   (FROZEN)
Fusion Classifier    :      363,523 params   (TRAINABLE)
------------------------------------------------------------
Total Fusion Model   :    6,372,003 params
Trainable Parameters :      364,419
Frozen Parameters    :    6,007,584


In [ ]:
def main():

    DATASET_PATH = "/content/drive/MyDrive/Dataset/castor_v2_224x224"  # Adjust path
    YOLO_WEIGHTS = '/content/YOLO26_YOLO26_Original_fold1.pth'  # Adjust path
    VIT_WEIGHTS = '/content/ViTPretrainedReducedFreeze_Improved_NoLabelSmoothing_fold5.pth'  # Adjust path
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.0001
    N_SPLITS = 5
    PATIENCE = 9

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load dataset
    dataset, class_names = get_dataset(DATASET_PATH)
    print(f"Dataset loaded: {len(dataset)} samples, {len(class_names)} classes")

    # Prepare indices and labels for stratified k-fold
    indices = np.arange(len(dataset))
    labels = np.array([label for _, label in dataset])

    # Stratified K-Fold
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

    fold_results = []

    print(f'\n{"="*60}')
    print(f'Starting {N_SPLITS}-Fold Cross-Validation for YOLO-ViT Fusion')
    print(f'{"="*60}\n')

    fold_results = []
    all_fold_data = []  # Store best metrics for final summary table

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        print(f'\n{"="*60}')
        print(f'FOLD {fold + 1}/{N_SPLITS}')
        print(f'{"="*60}')

        is_last_fold = (fold == N_SPLITS - 1)

        CONFIG = {
            'fold': fold + 1,
            'batch_size': BATCH_SIZE,
            'epochs': NUM_EPOCHS,
            'learning_rate': LEARNING_RATE,
            'model': 'YOLO26-ViT-Fusion',
            'patience': PATIENCE
        }

        # Initialize W&B for this fold
        run_name = f"YOLO26ViT_Fusion_Fold-{fold + 1}"
        wandb.init(
            project="YOLO26_ViT_Fusion_Classification",
            name=run_name,
            group="YOLO26ViT_Fusion_Original_2",
            config=CONFIG,
            reinit=True
        )

        # Calculate class weights for this fold
        class_counts = {}
        for idx in train_idx:
            _, label = dataset[idx]
            class_counts[label] = class_counts.get(label, 0) + 1
        total_samples = len(train_idx)
        num_classes = len(class_counts)
        weights = [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)]
        class_weights = torch.FloatTensor(weights).to(device)

        # Create data loaders
        train_dataset = Subset(dataset, train_idx)
        val_dataset = Subset(dataset, val_idx)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

        # Load backbones and build fusion model
        yolo_backbone, vit_backbone = load_pretrained_backbones(YOLO_WEIGHTS, VIT_WEIGHTS)
        fusion_model = YOLOViTFusion(
            yolo_backbone, vit_backbone, num_classes=len(class_names), dropout=0.3
        ).to(device)

        # Freeze backbones, train only classifier
        for param in fusion_model.yolo.parameters():
            param.requires_grad = False
        for param in fusion_model.vit.parameters():
            param.requires_grad = False

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(fusion_model.classifier.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)

        early_stopping = EarlyStopping(patience=PATIENCE, verbose=True)

        best_val_acc = 0.0
        best_val_loss = float('inf')  # Track best validation loss
        best_epoch = -1
        best_prec = best_rec = best_f1 = best_roc = 0
        best_val_cm = None
        best_train_cm = None
        best_val_report = None
        best_train_report = None
        best_model_path = f'YOLO26ViT_Fusion_fold{fold+1}.pth'

        for epoch in range(NUM_EPOCHS):
            print(f'\nEpoch [{epoch+1}/{NUM_EPOCHS}]')

            # Training
            fusion_model.train()
            train_loss, train_correct, train_total = 0.0, 0, 0
            train_all_labels, train_all_preds, train_all_probs = [], [], []
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = fusion_model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                probs = torch.softmax(outputs, dim=1)
                _, predicted = outputs.max(1)
                train_total += labels.size(0)
                train_correct += predicted.eq(labels).sum().item()

                train_all_labels.extend(labels.cpu().numpy())
                train_all_preds.extend(predicted.cpu().numpy())
                train_all_probs.extend(probs.detach().cpu().numpy())

            train_acc = 100. * train_correct / train_total
            train_loss /= len(train_loader)
            train_metrics = calculate_additional_metrics(train_all_labels, train_all_preds, np.array(train_all_probs), len(class_names))

            # Validation
            fusion_model.eval()
            val_loss, val_correct, val_total = 0.0, 0, 0
            val_all_labels, val_all_preds, val_all_probs = [], [], []
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = fusion_model(images)
                    loss = criterion(outputs, labels)
                    val_loss += loss.item()
                    probs = torch.softmax(outputs, dim=1)
                    _, predicted = outputs.max(1)
                    val_total += labels.size(0)
                    val_correct += predicted.eq(labels).sum().item()

                    val_all_labels.extend(labels.cpu().numpy())
                    val_all_preds.extend(predicted.cpu().numpy())
                    val_all_probs.extend(probs.cpu().numpy())

            val_acc = 100. * val_correct / val_total
            val_loss /= len(val_loader)
            val_metrics = calculate_additional_metrics(val_all_labels, val_all_preds, np.array(val_all_probs), len(class_names))

            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

            scheduler.step(val_acc)
            current_lr = optimizer.param_groups[0]['lr']
            print(f'Learning Rate: {current_lr:.6f}')

            # Log to W&B
            wandb.log({
                'epoch': epoch + 1,
                'learning_rate': current_lr,
                'train_loss': train_loss,
                'train_accuracy': train_acc,
                'train_precision': train_metrics['precision'],
                'train_recall': train_metrics['recall'],
                'train_f1': train_metrics['f1'],
                'train_auc': train_metrics['roc_auc'],
                'val_loss': val_loss,
                'val_accuracy': val_acc,
                'val_precision': val_metrics['precision'],
                'val_recall': val_metrics['recall'],
                'val_f1': val_metrics['f1'],
                'val_auc': val_metrics['roc_auc']
            })

            # Save model when validation loss improves
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_val_acc = val_acc
                best_prec = val_metrics['precision']
                best_rec = val_metrics['recall']
                best_f1 = val_metrics['f1']
                best_roc = val_metrics['roc_auc']
                best_epoch = epoch + 1

                torch.save(fusion_model.state_dict(), best_model_path)
                wandb.save(best_model_path)
                print(f'✓ Saved best model for fold {fold+1} (Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%)')

                # Store confusion matrices and classification reports
                best_val_cm = confusion_matrix(val_all_labels, val_all_preds)
                best_val_report = classification_report(val_all_labels, val_all_preds, target_names=class_names, output_dict=False)
                best_train_cm = confusion_matrix(train_all_labels, train_all_preds)
                best_train_report = classification_report(train_all_labels, train_all_preds, target_names=class_names, output_dict=False)

            early_stopping(val_acc)
            if early_stopping.early_stop:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break

        # Log confusion matrices and classification reports
        if best_val_cm is not None:
            val_cm_img = plot_cm(best_val_cm, class_names, f"Fold{fold+1}_Validation_CM")
            train_cm_img = plot_cm(best_train_cm, class_names, f"Fold{fold+1}_Training_CM")

            wandb.log({
                "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                "best_training_confusion_matrix": wandb.Image(train_cm_img),
                "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>"),
                "best_training_classification_report": wandb.Html(f"<pre>{best_train_report}</pre>")
            })

        # Store this fold's metrics in summary
        wandb.summary["fold"] = fold + 1
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_loss"] = best_val_loss
        wandb.summary["final_val_loss"] = best_val_loss
        wandb.summary["final_val_accuracy"] = best_val_acc
        wandb.summary["final_precision"] = best_prec
        wandb.summary["final_recall"] = best_rec
        wandb.summary["final_f1"] = best_f1
        wandb.summary["final_roc_auc"] = best_roc

        # Log summary table on last fold
        if is_last_fold:
            # Add current fold's data
            all_fold_data.append([
                fold + 1, best_epoch, best_val_acc, best_val_loss, best_prec, best_rec, best_f1, best_roc
            ])

            # Create summary table with ALL folds
            summary_table = wandb.Table(
                columns=["fold", "best_epoch", "best_accuracy", "best_loss", "best_precision", "best_recall", "best_f1", "best_roc_auc"]
            )
            for fold_data in all_fold_data:
                summary_table.add_data(*fold_data)
            wandb.log({"all_folds_best_metrics": summary_table})

        fold_results.append({
            'fold': fold + 1,
            'best_val_acc': best_val_acc,
            'best_epoch': best_epoch,
            'best_loss': best_val_loss,
            'best_prec': best_prec,
            'best_rec': best_rec,
            'best_f1': best_f1,
            'best_roc': best_roc
        })

        # Store data for folds 1-4 (fold 5 adds itself inside the is_last_fold block)
        if not is_last_fold:
            all_fold_data.append([
                fold + 1, best_epoch, best_val_acc, best_val_loss, best_prec, best_rec, best_f1, best_roc
            ])

        print(f'\nFold {fold + 1} Best Validation Accuracy: {best_val_acc:.2f}%')
        print(f'{"="*60}')

        wandb.finish()

    # Summary
    print(f'\n{"="*60}')
    print('K-FOLD CROSS-VALIDATION SUMMARY')
    print(f'{"="*60}')

    for result in fold_results:
        print(f"Fold {result['fold']}: Best Val Acc = {result['best_val_acc']:.2f}% (epoch {result['best_epoch']})")

    # Calculate averages
    avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
    avg_loss = np.mean([r['best_loss'] for r in fold_results])
    avg_prec = np.mean([r['best_prec'] for r in fold_results])
    avg_rec = np.mean([r['best_rec'] for r in fold_results])
    avg_f1 = np.mean([r['best_f1'] for r in fold_results])
    avg_roc = np.mean([r['best_roc'] for r in fold_results])

    print(f"\n====== FINAL K-FOLD RESULTS (AVG OVER ALL FOLDS) ======")
    print(f"Avg Val Acc  : {avg_acc:.4f}")
    print(f"Avg Val Loss : {avg_loss:.4f}")
    print(f"Avg Precision: {avg_prec:.4f}")
    print(f"Avg Recall   : {avg_rec:.4f}")
    print(f"Avg F1 Score : {avg_f1:.4f}")
    print(f"Avg ROC AUC  : {avg_roc:.4f}")
    print(f'{"="*60}')


main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
Using device: cuda
Found 991 images across 3 classes: ['Fresh Leafs', 'Semilooper', 'Spodoptera']
Dataset loaded: 991 samples, 3 classes

Starting 5-Fold Cross-Validation for YOLO-ViT Fusion


FOLD 1/5


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


...

Building YOLOv8 Classifier...
Loading YOLO weights from /content/YOLO26_YOLO26_Original_fold1.pth ...
✓ Loaded YOLO weights (remapped keys)
Building ViT-Tiny...
Loading ViT weights from /content/ViTPretrainedReducedFreeze_Improved_NoLabelSmoothing_fold5.pth ...
✓ Backbones ready
YOLO feature dim: 256
ViT feature dim: 192

Epoch [1/50]


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Train Loss: 0.3528, Train Acc: 95.84%
Val Loss: 0.1436, Val Acc: 100.00%
Learning Rate: 0.000100
✓ Saved best model for fold 5 (Val Acc: 100.00%)

Epoch [2/50]
Train Loss: 0.1390, Train Acc: 99.75%
Val Loss: 0.0838, Val Acc: 100.00%
Learning Rate: 0.000100

Epoch [3/50]
Train Loss: 0.1055, Train Acc: 99.50%
Val Loss: 0.0535, Val Acc: 100.00%
Learning Rate: 0.000100

Epoch [4/50]
Train Loss: 0.0835, Train Acc: 99.50%
Val Loss: 0.0370, Val Acc: 100.00%
Learning Rate: 0.000100

Epoch [5/50]
Train Loss: 0.0548, Train Acc: 99.75%
Val Loss: 0.0270, Val Acc: 100.00%
Learning Rate: 0.000050

Epoch [6/50]
Train Loss: 0.0553, Train Acc: 99.62%
Val Loss: 0.0218, Val Acc: 100.00%
Learning Rate: 0.000050

Epoch [7/50]
Train Loss: 0.0357, Train Acc: 99.75%
Val Loss: 0.0187, Val Acc: 100.00%
Learning Rate: 0.000050

Epoch [8/50]
Train Loss: 0.0396, Train Acc: 99.62%
Val Loss: 0.0201, Val Acc: 100.00%
Learning Rate: 0.000050

Epoch [9/50]
Train Loss: 0.0472, Train Acc: 99.62%
Val Loss: 0.0164, Val Acc

epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
learning_rate,████▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁█▇▇█▇▇▇▇▇▇███████▇██▇▇████████▇▇███▇▇█▇
train_auc,▁▆▆▆████▅█████████████▃██▂████▇█████████
train_f1,▁█▇▇██▇▇▇▇█▇███▇███▇██▇▇█▇█████▇▇███▇▇█▇
train_loss,█▃▃▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_precision,▁█▇▇▇▇▇▇▇▇█▇██▇███▇██▇▇██▇▇██▇██▇███▇▇█▇
train_recall,▁▇▇▇██▇▇▇▇█▇▇▇▇▇█▇▇█▇▇▇██▇▇██▇▇▇▇█▇█▇▇▇▇
val_accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_auc,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+4,...



K-FOLD CROSS-VALIDATION SUMMARY
Fold 1: Best Val Acc = 99.50% (epoch 1)
Fold 2: Best Val Acc = 100.00% (epoch 1)
Fold 3: Best Val Acc = 100.00% (epoch 1)
Fold 4: Best Val Acc = 99.49% (epoch 3)
Fold 5: Best Val Acc = 100.00% (epoch 1)

====== FINAL K-FOLD RESULTS (AVG OVER ALL FOLDS) ======
Avg Val Acc  : 99.7985
Avg Val Loss : 0.1247
Avg Precision: 0.9967
Avg Recall   : 0.9979
Avg F1 Score : 0.9973
Avg ROC AUC  : 1.0000


In [ ]:
class ClassifierWrapper(nn.Module):
    """Wraps the classifier (nn.Sequential) so DeepExplainer can call it directly
    Input: tensor of shape (N, fused_dim)
    Output: tensor of shape (N, num_classes) (logits)
    """
    def __init__(self, classifier: nn.Module):
        super().__init__()
        self.classifier = classifier

    def forward(self, x):
        out = self.classifier(x)
        # Clone to avoid view+inplace issues with autograd hooks (SHAP)
        return out.clone()


def compute_embeddings(backbone_yolo, backbone_vit, loader, max_samples=None, device=device):
    """Compute YOLO and ViT embeddings for images in loader. Returns numpy arrays.
    Features are L2 normalized to match fusion model processing.
    Returns: yolo_feats (N, y_dim), vit_feats (N, v_dim), labels (N,)
    """
    y_list = []
    v_list = []
    labs = []
    cnt = 0

    backbone_yolo.eval()
    backbone_vit.eval()

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            f_y = backbone_yolo(batch_X)
            f_v = backbone_vit(batch_X)

            # L2 normalize features to match fusion model processing
            f_y = torch.nn.functional.normalize(f_y, p=2, dim=1)
            f_v = torch.nn.functional.normalize(f_v, p=2, dim=1)

            y_list.append(f_y.detach().cpu().numpy())
            v_list.append(f_v.detach().cpu().numpy())
            labs.append(batch_y.numpy())

            cnt += batch_X.size(0)
            if (max_samples is not None) and (cnt >= max_samples):
                break

    y_arr = np.concatenate(y_list, axis=0)
    v_arr = np.concatenate(v_list, axis=0)
    labs_arr = np.concatenate(labs, axis=0)

    if (max_samples is not None) and (y_arr.shape[0] > max_samples):
        y_arr = y_arr[:max_samples]
        v_arr = v_arr[:max_samples]
        labs_arr = labs_arr[:max_samples]

    return y_arr, v_arr, labs_arr


In [ ]:
def run_shap_on_fusion(
    yolo_backbone,
    vit_backbone,
    fusion_classifier,
    background_feats,
    test_feats,
    class_names,
    device=device,
    y_dim: int = None,
    v_dim: int = None,
    explainer_method: str = 'auto',  # 'deep' | 'gradient' | 'auto'
):
    """Run SHAP explainer on classifier-only using background and test fused features.

    background_feats, test_feats: numpy arrays (N, fused_dim)
    Returns dictionary with raw shap arrays and branch-aggregated results.
    """
    # Build classifier wrapper and move to device
    classifier = ClassifierWrapper(fusion_classifier).to(device)
    classifier.eval()

    # Convert background and test features to torch tensors
    background_t = torch.from_numpy(background_feats.astype(np.float32)).to(device)
    test_t = torch.from_numpy(test_feats.astype(np.float32)).to(device)

    # Determine branch sizes (prefer explicit args derived from actual feature arrays)
    if y_dim is None:
        y_dim = yolo_backbone.out_dim
    if v_dim is None:
        v_dim = vit_backbone.out_dim
    fused_expected = y_dim + v_dim

    # Helper to compute shap array given an explainer builder
    def _compute_shap_with(explainer_builder, use_name):
        print(f"Building SHAP {use_name} (this may take a moment)...")
        explainer = explainer_builder(classifier, background_t)
        print(f"Computing SHAP values for test set using {use_name}...")
        shap_values = explainer.shap_values(test_t)

        # Normalize various possible SHAP output formats into (num_classes, n_test, fused_dim)
        num_classes = len(class_names)

        # Convert to numpy array if necessary
        try:
            if isinstance(shap_values, list):
                # Try stacking the list
                stacked = np.stack([np.asarray(sv) for sv in shap_values], axis=0)
            else:
                stacked = np.asarray(shap_values)
        except Exception as e:
            print(f"Error converting SHAP output to array: {e}")
            raise

        print(f"-> Raw SHAP output shape: {stacked.shape}")

        s = stacked.shape
        # Try to detect and permute axes to target (num_classes, n_test, fused_dim)
        if stacked.ndim == 3:
            # Common possibilities:
            # (num_classes, n_test, fused_dim)
            if s[0] == num_classes and s[2] == fused_expected:
                shap_arr_local = stacked
            # (n_test, fused_dim, num_classes)
            elif s[0] == test_t.shape[0] and s[2] == num_classes:
                shap_arr_local = stacked.transpose(2, 0, 1)
            # (n_test, num_classes, fused_dim)
            elif s[0] == test_t.shape[0] and s[1] == num_classes:
                shap_arr_local = stacked.transpose(1, 0, 2)
            # (num_classes, fused_dim, n_test)
            elif s[0] == num_classes and s[1] == fused_expected and s[2] == test_t.shape[0]:
                shap_arr_local = stacked.transpose(0, 2, 1)
            # (fused_dim, n_test, num_classes)
            elif s[0] == fused_expected and s[2] == num_classes:
                shap_arr_local = stacked.transpose(2, 1, 0)
            else:
                # Unknown ordering; attempt heuristics
                # If one dimension matches fused_expected, assume it's fused_dim
                fused_axis = None
                for i, dim in enumerate(s):
                    if dim == fused_expected:
                        fused_axis = i
                if fused_axis is not None:
                    # Move fused axis to last and num_classes to first
                    perm = list(range(len(s)))
                    perm.pop(fused_axis)
                    perm.append(fused_axis)
                    reordered = stacked.transpose(perm)
                    # Now try to get (num_classes, n_test, fused_dim)
                    if reordered.shape[0] == num_classes:
                        shap_arr_local = reordered
                    else:
                        # Fallback attempt: transpose to (num_classes, n_test, fused_dim) by rotating axes
                        shap_arr_local = reordered.transpose(2, 0, 1) if reordered.ndim == 3 else None
                else:
                    shap_arr_local = None
        elif stacked.ndim == 2:
            # Possibly single class or collapsed; try to reshape to (num_classes, n_test, fused_dim)
            # if fused_expected divides second dim
            if stacked.shape[1] == fused_expected and num_classes == 1:
                shap_arr_local = stacked.reshape((1, stacked.shape[0], stacked.shape[1]))
            else:
                shap_arr_local = None
        else:
            shap_arr_local = None

        if shap_arr_local is None:
            raise RuntimeError(f"Unable to normalize SHAP output shape {s} into (num_classes, n_test, fused_dim={fused_expected}).")

        print(f"-> {use_name} normalized to shap array with shape: {shap_arr_local.shape}")
        return shap_arr_local

    shap_arr = None
    last_error = None

    # Try preferred method(s)
    methods = []
    if explainer_method == 'auto':
        methods = ['deep', 'gradient']
    else:
        methods = [explainer_method]

    for method in methods:
        try:
            if method == 'deep':
                shap_arr = _compute_shap_with(shap.DeepExplainer, 'DeepExplainer')
            elif method == 'gradient':
                shap_arr = _compute_shap_with(shap.GradientExplainer, 'GradientExplainer')
            else:
                raise ValueError(f"Unknown explainer method: {method}")

            # Validate feature axis length
            if shap_arr.ndim == 3 and shap_arr.shape[2] == fused_expected:
                # success
                used_method = method
                break
            else:
                # Not matching expected shape — record and try next
                last_error = (
                    f"Explainer '{method}' returned incompatible shape {shap_arr.shape}, "
                    f"expected (num_classes, n_test, fused_dim={fused_expected})."
                )
                print("WARNING:", last_error)
                shap_arr = None
        except Exception as e:
            last_error = str(e)
            print(f"Explainer '{method}' failed with error: {e}")
            shap_arr = None

    if shap_arr is None:
        # Provide helpful diagnostic and raise
        raise RuntimeError(
            "SHAP explainers failed to produce expected feature-wise attributions.\n"
            f"Background feats shape: {background_feats.shape}, test feats shape: {test_feats.shape}.\n"
            f"YOLO backbone out_dim={yolo_backbone.out_dim}, ViT backbone out_dim={vit_backbone.out_dim}.\n"
            f"Last explainer error: {last_error}\n"
            "Try using GradientExplainer explicitly or verify that the inputs passed are (N, fused_dim) arrays."
        )

    # At this point shap_arr has shape (num_classes, n_test, fused_dim)
    used_method = locals().get('used_method', 'unknown')
    print(f"Using SHAP method: {used_method} — shap array shape: {shap_arr.shape}")

    y_ind = np.arange(0, y_dim)
    v_ind = np.arange(y_dim, y_dim + v_dim)

    # Aggregate to branch-level
    num_classes = shap_arr.shape[0]
    n_test = shap_arr.shape[1]

    branch_signed = np.zeros((num_classes, n_test, 2), dtype=float)  # (class, sample, branch)
    branch_abs = np.zeros_like(branch_signed)

    for c in range(num_classes):
        vals = shap_arr[c]  # (n_test, fused_dim)
        branch_signed[c, :, 0] = vals[:, y_ind].sum(axis=1)
        branch_signed[c, :, 1] = vals[:, v_ind].sum(axis=1)

        branch_abs[c, :, 0] = np.abs(vals[:, y_ind]).sum(axis=1)
        branch_abs[c, :, 1] = np.abs(vals[:, v_ind]).sum(axis=1)

    # Compute means across samples for each class
    mean_signed_per_class = branch_signed.mean(axis=1)  # (num_classes, 2)
    mean_abs_per_class = branch_abs.mean(axis=1)

    # Also compute overall mean absolute contribution across all classes and samples
    overall_mean_abs = branch_abs.mean(axis=(0, 1))  # (2,)

    results = {
        'shap_arr': shap_arr,
        'branch_signed': branch_signed,
        'branch_abs': branch_abs,
        'mean_signed_per_class': mean_signed_per_class,
        'mean_abs_per_class': mean_abs_per_class,
        'overall_mean_abs': overall_mean_abs,
    }

    return results


def save_and_plot_results(results, class_names, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    mean_signed = results['mean_signed_per_class']
    mean_abs = results['mean_abs_per_class']

    # DataFrame per class
    rows = []
    for i, cname in enumerate(class_names):
        rows.append({
            'class': cname,
            'yolo_mean_signed': mean_signed[i, 0],
            'vit_mean_signed': mean_signed[i, 1],
            'yolo_mean_abs': mean_abs[i, 0],
            'vit_mean_abs': mean_abs[i, 1],
        })

    df = pd.DataFrame(rows)
    csv_path = os.path.join(out_dir, 'shap_branch_summary_per_class.csv')
    df.to_csv(csv_path, index=False)
    print(f"Saved per-class SHAP summary to: {csv_path}")

    # Overall pie chart showing relative mean absolute contribution
    overall = results['overall_mean_abs']
    labels = ['YOLO', 'ViT']
    sizes = overall

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=['#4E79A7', '#F28E2B'])
    ax.set_title('YOLO vs ViT mean absolute SHAP contribution (overall)')
    pie_path = os.path.join(out_dir, 'shap_fusion_pie.png')
    fig.savefig(pie_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved YOLO vs ViT pie chart to: {pie_path}")

    return csv_path, pie_path


In [ ]:
def sample_balanced_indices(dataset, num_samples):
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    classes = np.unique(labels)
    per_class = num_samples // len(classes)

    indices = []
    for c in classes:
        cls_idx = np.where(labels == c)[0]
        chosen = np.random.choice(cls_idx, per_class, replace=False)
        indices.extend(chosen.tolist())

    return np.array(indices)

def main2(
    dataset_path: str = '/content/drive/MyDrive/Dataset/castor_v3_224x224',
    fusion_weights: str = '/content/YOLO26ViT_Fusion_best.pth',
    yolo_weights: str = '/content/YOLOv8normal_simple.pth',
    vit_weights: str  = '/content/vit_pretrained_simple (1).pth',
    num_background: int = 100,
    num_test: int = 64,
    batch_size: int = 8,
    out_dir: str = 'shap_results',
    test_start: int = 0,
):
    import os
    import numpy as np
    import pandas as pd
    import torch
    from torch.utils.data import DataLoader, Subset

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --------------------------------------------------
    # 1. Dataset
    # --------------------------------------------------
    dataset, class_names = get_dataset(dataset_path)
    num_classes = len(class_names)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # --------------------------------------------------
    # 2. Load backbones WITH CORRECT WEIGHTS
    # --------------------------------------------------
    yolo_backbone, vit_backbone = load_pretrained_backbones(
        yolo_weights=yolo_weights,
        vit_weights=vit_weights,
    )

    # --------------------------------------------------
    # 3. Build fusion model and load fusion weights
    # --------------------------------------------------
    fusion_model = YOLOViTFusion(
        yolo_backbone=yolo_backbone,
        vit_backbone=vit_backbone,
        num_classes=num_classes,
        dropout=0.0  # eval mode
    ).to(device)

    fusion_model.eval()

    print(f"Loading fusion checkpoint: {fusion_weights}")
    ckpt = torch.load(fusion_weights, map_location=device)
    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt:
            fusion_model.load_state_dict(ckpt['model_state_dict'])
        elif 'state_dict' in ckpt:
            fusion_model.load_state_dict(ckpt['state_dict'])
        else:
            fusion_model.load_state_dict(ckpt)
    else:
        fusion_model.load_state_dict(ckpt)
    print("✓ Loaded fusion weights")

    fusion_classifier = fusion_model.classifier

        # --------------------------------------------------
    # 4. Balanced BACKGROUND sampling
    # --------------------------------------------------
    print(f"Sampling {num_background} background images (class-balanced)...")
    bg_indices = sample_balanced_indices(dataset, num_background)
    bg_subset = Subset(dataset, bg_indices)

    bg_loader = DataLoader(
        bg_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    y_bg, v_bg, _ = compute_embeddings(
        yolo_backbone,
        vit_backbone,
        bg_loader,
        max_samples=num_background
    )


    fused_bg = np.concatenate([y_bg, v_bg], axis=1)


    # --------------------------------------------------
    # 5. Balanced TEST sampling
    # --------------------------------------------------
    print(f"Sampling {num_test} test images (class-balanced)...")
    test_indices = sample_balanced_indices(dataset, num_test)
    test_subset = Subset(dataset, test_indices)

    test_loader = DataLoader(
        test_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    y_test, v_test, test_labels = compute_embeddings(
        yolo_backbone,
        vit_backbone,
        test_loader,
        max_samples=num_test
    )


    fused_test = np.concatenate([y_test, v_test], axis=1)

    # --------------------------------------------------
    # 8. RUN SHAP (DeepExplainer)
    # --------------------------------------------------
    results = run_shap_on_fusion(
        yolo_backbone=yolo_backbone,
        vit_backbone=vit_backbone,
        fusion_classifier=fusion_classifier,
        background_feats=fused_bg,
        test_feats=fused_test,
        class_names=class_names,
        y_dim=y_bg.shape[1],
        v_dim=v_bg.shape[1],
    )

    # --------------------------------------------------
    # 9. Save + print results
    # --------------------------------------------------
    csv_path, pie_path = save_and_plot_results(
        results,
        class_names,
        out_dir
    )

    df = pd.read_csv(csv_path)

    print("\nPer-class SHAP branch summary:")
    print(df.to_string(index=False))

    print("\nMean absolute contribution per class (YOLO, ViT):")
    for i, cname in enumerate(class_names):
        print(
            f"  Class '{cname}': "
            f"YOLO={df.loc[i,'yolo_mean_abs']:.6f}, "
            f"ViT={df.loc[i,'vit_mean_abs']:.6f}"
        )

    print("\n✓ SHAP analysis completed correctly.")
main2()

Found 752 images across 3 classes: ['Fresh Leafs', 'Semilooper', 'Spodoptera']
Building YOLOv8 Classifier...
Loading YOLO weights from /content/YOLOv8normal_simple.pth ...
✓ Loaded YOLO weights (remapped keys)
Building ViT-Tiny...
Loading ViT weights from /content/vit_pretrained_simple (1).pth ...
✓ Backbones ready
YOLO feature dim: 256
ViT feature dim: 192
Loading fusion checkpoint: /content/YOLO26ViT_Fusion_best.pth
✓ Loaded fusion weights
Sampling 100 background images (class-balanced)...
Sampling 64 test images (class-balanced)...
Building SHAP DeepExplainer (this may take a moment)...
Computing SHAP values for test set using DeepExplainer...
-> Raw SHAP output shape: (63, 448, 3)
-> DeepExplainer normalized to shap array with shape: (3, 63, 448)
Using SHAP method: deep — shap array shape: (3, 63, 448)
Saved per-class SHAP summary to: shap_results/shap_branch_summary_per_class.csv
Saved YOLO vs ViT pie chart to: shap_results/shap_fusion_pie.png

Per-class SHAP branch summary:
     